In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# Project path
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent.parent

sys.path.append(
    str(PROJECT_ROOT / "src")
)

# ------------------------------------------------------------
# OpenETBench imports
# ------------------------------------------------------------

from extraction.gee import initialize
from extraction.sites import get_site
from extraction.products import (
    get_product,
    list_products,
)

from extraction.extractor import (
    extract_timeseries,
)

from harmonization.temporal import (
    align_to_common_dates,
)

from harmonization.merge import (
    merge_observed_satellite,
)

from benchmarking.metrics import (
    calculate_metrics,
)

from utils.io import (
    load_bharatflux,
)

from preprocessing.exporter import (
    export_results,
)

# ------------------------------------------------------------
# Initialize GEE
# ------------------------------------------------------------

initialize()

# print("✓ Earth Engine initialized successfully.")

✓ Earth Engine initialized successfully.
✓ Earth Engine initialized successfully.


In [3]:
# ============================================================
# Sprint 2 configuration
# ============================================================

SITE_ID = "BFT"

START_DATE = "2016-01-01"
END_DATE = "2016-12-31"

PRODUCTS = [
    "MOD16A2GF",
    "ERA5-LAND",
    "FLDAS",
    "GLDAS",
    "MERRA2",
    "PMLV2",
]

print("Site:", SITE_ID)
print("Period:", START_DATE, "→", END_DATE)
print("Products:", PRODUCTS)

Site: BFT
Period: 2016-01-01 → 2016-12-31
Products: ['MOD16A2GF', 'ERA5-LAND', 'FLDAS', 'GLDAS', 'MERRA2', 'PMLV2']


In [4]:
print("Registered ET products:")
print(list_products())

print("\nSprint 2 products:")

for name in PRODUCTS:
    product = get_product(name)

    print(
        f"✓ {name:12s} → "
        f"{product.name:10s} | "
        f"{product.temporal_resolution:8s} | "
        f"{product.spatial_resolution} m"
    )

Registered ET products:
['ERA5-LAND', 'FLDAS', 'GLDAS', 'MERRA2', 'MOD16A2GF', 'PMLV2', 'SSEBOP']

Sprint 2 products:
✓ MOD16A2GF    → MOD16A2GF  | 8-day    | 500 m
✓ ERA5-LAND    → ERA5-Land  | Daily    | 11132 m
✓ FLDAS        → FLDAS      | Monthly  | 11100 m
✓ GLDAS        → GLDAS      | Daily    | 27830 m
✓ MERRA2       → MERRA2     | Hourly   | 55000 m
✓ PMLV2        → PML-V2     | 8-day    | 500 m


In [5]:
# ============================================================
# Load processed BharatFlux data
# ============================================================

processed_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bharatflux"
)

datasets = load_bharatflux(
    processed_dir
)

print("Available processed BharatFlux datasets:")
for name in datasets:
    print(" ", name)

Available processed BharatFlux datasets:
  BFT_2016_LE_ET_dmean
  BFT_2017_LE_ET_dmean
  BFT_2018_LE_ET_dmean
  BIT_2016_LE_ET_dmean
  BIT_2017_LE_ET_dmean
  BKC_2014
  BKC_2015
  BKC_2016
  DIT_2016_LE_ET_dmean
  DIT_2017_LE_ET_dmean
  JIT_2016_ET_LE_dmean
  JIT_2017_ET_LE_dmean
  JIT_2018_ET_LE_dmean
  KKM_2014_LE_ET_dmean
  KKM_2015_LE_ET_dmean
  KKM_2016_LE_ET_dmean
  KNP_2016
  KNP_2017
  KNP_2018
  NIT_2016_ET_LE_dmean
  NIT_2017_ET_LE_dmean
  NIT_2018_ET_LE_dmean
  PVM_2017
  SFT_2014_LE_ET_dmean
  SFT_2015_LE_ET_dmean
  SFT_2016_LE_ET_dmean
  SIT_2016_LE_ET_dmean
  SIT_2017_LE_ET_dmean
  SIT_2018_LE_ET_dmean
  UIT_2016_LE_ET_dmean
  UIT_2017_LE_ET_dmean


In [6]:
# ------------------------------------------------------------
# Select BFT 2016
# ------------------------------------------------------------

observed_dataset = datasets["BFT_2016_LE_ET_dmean"]

observed = observed_dataset.data.copy()

print("Observed dataset:")
print(observed.shape)

display(
    observed.head()
)

Observed dataset:
(366, 3)


,DoY,LE,ET
0,1,121.533229,4.666875
1,2,124.119004,4.766169
2,3,104.577071,4.015759
3,4,109.743590,4.214153
4,5,110.055843,4.226144


In [7]:
RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / SITE_ID
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Results directory:")
print(RESULTS_DIR)

Results directory:
e:\GRAVITY\IITM-Pune\OpenETBench\results\BFT


In [8]:
# ============================================================
# SPRINT 2 — MULTI-PRODUCT VALIDATION
# ============================================================

results = {}

for product_name in PRODUCTS:

    print("\n" + "=" * 70)
    print(f"TESTING PRODUCT: {product_name}")
    print("=" * 70)

    try:

        # ----------------------------------------------------
        # 1. Get product configuration
        # ----------------------------------------------------

        product = get_product(
            product_name
        )

        print("\nProduct:")
        print(product)

        # ----------------------------------------------------
        # 2. Extract through common API
        # ----------------------------------------------------

        print("\nExtracting satellite ET...")

        satellite = extract_timeseries(
            get_site(SITE_ID),
            product,
            START_DATE,
            END_DATE,
        )

        print(
            "Satellite shape:",
            satellite.shape,
        )

        display(
            satellite.head()
        )

        # ----------------------------------------------------
        # 3. Temporal harmonization
        # ----------------------------------------------------

        observed_aligned, satellite_aligned = (
            align_to_common_dates(
                observed,
                satellite,
            )
        )

        print(
            "\nAligned observations:",
            observed_aligned.shape,
        )

        print(
            "Aligned satellite:",
            satellite_aligned.shape,
        )

        # ----------------------------------------------------
        # 4. Merge
        # ----------------------------------------------------

        merged = merge_observed_satellite(
            observed_aligned,
            satellite_aligned,
        )

        print(
            "\nBenchmark dataframe:",
            merged.shape,
        )

        display(
            merged.head()
        )

        # ----------------------------------------------------
        # 5. Benchmark
        # ----------------------------------------------------

        metrics = calculate_metrics(
            merged
        )

        print("\nBenchmark statistics")
        print("--------------------")
        print(
            f"RMSE        : {metrics.rmse:.4f}"
        )
        print(
            f"MAE         : {metrics.mae:.4f}"
        )
        print(
            f"Bias        : {metrics.bias:.4f}"
        )
        print(
            f"Correlation : {metrics.correlation:.4f}"
        )
        print(
            f"R²          : {metrics.r2:.4f}"
        )

        # ----------------------------------------------------
        # 6. Export
        # ----------------------------------------------------

        product_dir = (
            RESULTS_DIR
            / product_name
        )

        extraction_path, benchmark_path = (
            export_results(
                merged,
                metrics,
                product_dir,
            )
        )

        print("\nExports:")
        print(
            "✓ extraction.csv:",
            extraction_path,
        )
        print(
            "✓ benchmark.json:",
            benchmark_path,
        )

        # ----------------------------------------------------
        # 7. Store validation result
        # ----------------------------------------------------

        results[product_name] = {
            "status": "PASSED",
            "n": len(merged),
            "rmse": metrics.rmse,
            "mae": metrics.mae,
            "bias": metrics.bias,
            "correlation": metrics.correlation,
            "r2": metrics.r2,
        }

        print(
            f"\n✓ {product_name} PASSED"
        )

    except Exception as exc:

        results[product_name] = {
            "status": "FAILED",
            "error": str(exc),
        }

        print(
            f"\n✗ {product_name} FAILED"
        )

        print(
            "Error:",
            repr(exc),
        )


TESTING PRODUCT: MOD16A2GF

Product:
ETProduct(name='MOD16A2GF', collection='MODIS/061/MOD16A2GF', band='ET', scale_factor=0.1, spatial_resolution=500, temporal_resolution='8-day', units='mm/8-day', provider='NASA', coverage='Global', product_type='Remote Sensing', aggregation='native', sampling='mean')

Extracting satellite ET...
Satellite shape: (46, 3)


,Date,DoY,ET
0,2016-01-01,1,3.793601
1,2016-01-09,9,3.947705
2,2016-01-17,17,4.724370
3,2016-01-25,25,2.116354
4,2016-02-02,33,2.866710



Aligned observations: (46, 3)
Aligned satellite: (46, 3)

Benchmark dataframe: (46, 5)


,Date,DoY,Observed_LE,Observed_ET,Satellite_ET
0,2016-01-01,1,121.533229,4.666875,3.793601
1,2016-01-09,9,107.454253,4.126243,3.947705
2,2016-01-17,17,134.438082,5.162422,4.724370
3,2016-01-25,25,87.810117,3.371908,2.116354
4,2016-02-02,33,64.205003,2.465472,2.866710



Benchmark statistics
--------------------
RMSE        : 13.1718
MAE         : 8.9248
Bias        : 7.6164
Correlation : 0.7880
R²          : 0.6210

Exports:
✓ extraction.csv: e:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\MOD16A2GF\extraction.csv
✓ benchmark.json: e:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\MOD16A2GF\benchmark.json

✓ MOD16A2GF PASSED

TESTING PRODUCT: ERA5-LAND

Product:
ETProduct(name='ERA5-Land', collection='ECMWF/ERA5_LAND/DAILY_AGGR', band='total_evaporation_sum', scale_factor=-1000.0, spatial_resolution=11132, temporal_resolution='Daily', units='mm/day', provider='ECMWF', coverage='Global', product_type='Reanalysis', aggregation='native', sampling='mean')

Extracting satellite ET...
Satellite shape: (365, 3)


...


Benchmark statistics
--------------------
RMSE        : 3.5691
MAE         : 3.0988
Bias        : -3.0751
Correlation : 0.7730
R²          : 0.5975

Exports:
✓ extraction.csv: e:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\MERRA2\extraction.csv
✓ benchmark.json: e:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\MERRA2\benchmark.json

✓ MERRA2 PASSED

TESTING PRODUCT: PMLV2

Product:
ETProduct(name='PML-V2', collection='projects/pml_evapotranspiration/PML/OUTPUT/PML_V22a', band='ET', scale_factor=0.01, spatial_resolution=500, temporal_resolution='8-day', units='mm/8-day', provider='PML', coverage='Global', product_type='Remote Sensing', aggregation='native', sampling='buffer')

Extracting satellite ET...
Satellite shape: (46, 3)


,Date,DoY,ET
0,2016-01-01,1,0.860227
1,2016-01-09,9,1.048004
2,2016-01-17,17,0.936115
3,2016-01-25,25,1.055499
4,2016-02-02,33,1.041124



Aligned observations: (46, 3)
Aligned satellite: (46, 3)

Benchmark dataframe: (46, 5)


,Date,DoY,Observed_LE,Observed_ET,Satellite_ET
0,2016-01-01,1,121.533229,4.666875,0.860227
1,2016-01-09,9,107.454253,4.126243,1.048004
2,2016-01-17,17,134.438082,5.162422,0.936115
3,2016-01-25,25,87.810117,3.371908,1.055499
4,2016-02-02,33,64.205003,2.465472,1.041124



Benchmark statistics
--------------------
RMSE        : 4.2034
MAE         : 3.5289
Bias        : -3.4798
Correlation : 0.6341
R²          : 0.4020

Exports:
✓ extraction.csv: e:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\PMLV2\extraction.csv
✓ benchmark.json: e:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\PMLV2\benchmark.json

✓ PMLV2 PASSED


In [9]:
# ============================================================
# SPRINT 2 VALIDATION REPORT
# ============================================================

print("\n")
print("=" * 70)
print("SPRINT 2 VALIDATION REPORT")
print("=" * 70)

for product_name, result in results.items():

    if result["status"] == "PASSED":

        print(
            f"✓ {product_name:12s} "
            f"PASSED | "
            f"n={result['n']:3d} | "
            f"RMSE={result['rmse']:.4f} | "
            f"R²={result['r2']:.4f}"
        )

    else:

        print(
            f"✗ {product_name:12s} FAILED | "
            f"{result['error']}"
        )

passed = sum(
    r["status"] == "PASSED"
    for r in results.values()
)

failed = len(results) - passed

print("\n" + "-" * 70)

print(
    f"Passed: {passed}/{len(PRODUCTS)}"
)

print(
    f"Failed: {failed}/{len(PRODUCTS)}"
)

if failed == 0:

    print(
        "\n🎉 SPRINT 2 PASSED."
    )

else:

    print(
        "\n⚠ SPRINT 2 NOT YET PASSED."
    )

    print(
        "Fix failed products before proceeding."
    )



SPRINT 2 VALIDATION REPORT
✓ MOD16A2GF    PASSED | n= 46 | RMSE=13.1718 | R²=0.6210
✓ ERA5-LAND    PASSED | n=365 | RMSE=3.9014 | R²=0.6220
✓ FLDAS        PASSED | n= 12 | RMSE=2.4809 | R²=0.7027
✓ GLDAS        PASSED | n=365 | RMSE=3.3363 | R²=0.6835
✓ MERRA2       PASSED | n=365 | RMSE=3.5691 | R²=0.5975
✓ PMLV2        PASSED | n= 46 | RMSE=4.2034 | R²=0.4020

----------------------------------------------------------------------
Passed: 6/6
Failed: 0/6

🎉 SPRINT 2 PASSED.
